In [ ]:
import sys
sys.path.insert(0, "../..")

from src.models        import LogDeepONet, build_logdeeponet
from src.data          import ODEIterableDataset, DirichletSampler
from src.physics       import Robertson
from src.training      import Trainer, build_dataloaders

import numpy as np
import matplotlib.pyplot as plt 
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Initialize Roberston Model
k1 = 4e-2
k2 = 3e7
k3 = 1e4
system = Robertson([k1, k2, k3])

# Initialize Sampler Object 
sampler = DirichletSampler(alpha=[1, 1, 1])

In [3]:
# Initialize Training and validation datasets
train_dataset    = ODEIterableDataset(size = 1000,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = (0, 1e4),
                                      method       = "BDF",
                                      output_mask  = None)

val_dataset      = ODEIterableDataset(size = 10,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = (0, 1e4),
                                      method       = "BDF", 
                                      output_mask  = None)

In [5]:
# Set Up DeepONet configuration 

DEEPONET_CONFIG = {
    
    "hidden_size" : 64,
    "depth"       : 4,
    "latent_size" : 120,
    "input_size_b": 3,
    "input_size_t": 1,
    "output_size" : 3,
    "activation"  : "tanh",

}

# Initialize DeepONet network 

deeponet = build_logdeeponet(DEEPONET_CONFIG)

In [6]:
# Set Up training Configuration as well as optimizer and loss functions

TRAIN_CONFIG = {

    "num_epochs"     : 80,
    "learning_rate"  : 0.00003501486406520061,
    "batch_size"     : 16,
    "num_workers"    : 2,
    "Save_model"     : True,
    "Save_directory" : "../../weights/best_robertson.pth"
}

optimizer = torch.optim.Adam
loss_fn   = torch.nn.MSELoss()

In [7]:
# Initialize Train Object and Train 

trainer   = Trainer(
    model         = deeponet,
    train_dataset = train_dataset, 
    val_dataset   = val_dataset, 
    optimizer     = optimizer,
    loss_fn       = loss_fn, 
    train_cfg     = TRAIN_CONFIG,
    model_cfg     = DEEPONET_CONFIG
)

trainer.run()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/enricp/.netrc.
wandb: Currently logged in as: nikpursals to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training:  24%|██▍       | 19/80 [03:23<10:43, 10.54s/it]/home/enricp/miniconda3/envs/physics/lib/python3.12/site-packages/scipy/integrate/_ivp/bdf.py:415: RuntimeWarning: invalid value encountered in subtract
  D[order + 2] = d - D[order + 1]
Training:  96%|█████████▋| 77/80 [13:58<00:32, 10.81s/it]/home/enricp/miniconda3/envs/physics/lib/python3.12/site-packages/scipy/integrate/_ivp/bdf.py:415: RuntimeWarning: invalid value encountered in subtract
  D[order + 2] = d - D[order + 1]
Training: 100%|██████████| 80/80 [14:30<00:00, 10.89s/it]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
